In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("int_sandbox")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "1g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 17:14:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 17:14:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 37676)
Traceback (most recent call last):
  File "/usr/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.8/socketserver.py", line 747, in __init__
    self.handle()
  Fil

In [2]:
# DataFrame 1: Orders
orders_data = [
    (1, "2024-05-01", 100.0, 1),
    (2, "2024-05-02", 150.0, 2),
    (3, "2024-05-03", 200.0, 3),
    (4, "2024-05-04", 300.0, 5),
    (5, "2024-05-10", 120.0, 2),
    (6, "2024-05-12", 250.0, 2)
]
orders_columns = ["order_id", "order_date", "amount", "customer_id"]
df_orders = spark.createDataFrame(orders_data, orders_columns)

In [3]:
df_orders.show(5, truncate=False)

+--------+----------+------+-----------+
|order_id|order_date|amount|customer_id|
+--------+----------+------+-----------+
|1       |2024-05-01|100.0 |1          |
|2       |2024-05-02|150.0 |2          |
|3       |2024-05-03|200.0 |3          |
|4       |2024-05-04|300.0 |5          |
|5       |2024-05-10|120.0 |2          |
+--------+----------+------+-----------+
only showing top 5 rows



In [4]:
# DataFrame 2: Customers
customers_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (5, "Diana"),
    (6, "Maria")
]
customers_columns = ["customer_id", "customer_name"]
df_customers = spark.createDataFrame(customers_data, customers_columns)


In [5]:
df_customers.show(5, truncate=False)

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|1          |Alice        |
|2          |Bob          |
|3          |Charlie      |
|5          |Diana        |
|6          |Maria        |
+-----------+-------------+



In [6]:
enriched_orders = (
    df_orders.alias("ord").join(df_customers.alias("cus"), F.col("ord.customer_id")==F.col("cus.customer_id"), "left").select("ord.*", "cus.customer_name")
).withColumn("bonus", F.col("amount") * 0.1)
enriched_orders.show(5, truncate=False)

+--------+----------+------+-----------+-------------+-----+
|order_id|order_date|amount|customer_id|customer_name|bonus|
+--------+----------+------+-----------+-------------+-----+
|2       |2024-05-02|150.0 |2          |Bob          |15.0 |
|1       |2024-05-01|100.0 |1          |Alice        |10.0 |
|3       |2024-05-03|200.0 |3          |Charlie      |20.0 |
|4       |2024-05-04|300.0 |5          |Diana        |30.0 |
|5       |2024-05-10|120.0 |2          |Bob          |12.0 |
+--------+----------+------+-----------+-------------+-----+
only showing top 5 rows



In [7]:
enriched_orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- bonus: double (nullable = true)



In [8]:
@udf
def order_classifier(amount: int) -> StringType():
    if amount < 150:
        return "low"
    elif 150 <= amount < 250:
        return "medium"
    elif amount >= 250:
        return "high"
    else:
        return "invalid amount"

In [9]:
enriched_orders = (
    enriched_orders.withColumn("amount_category", order_classifier(F.col("amount")))
)

enriched_orders.show(5, truncate=False)

+--------+----------+------+-----------+-------------+-----+---------------+
|order_id|order_date|amount|customer_id|customer_name|bonus|amount_category|
+--------+----------+------+-----------+-------------+-----+---------------+
|2       |2024-05-02|150.0 |2          |Bob          |15.0 |medium         |
|1       |2024-05-01|100.0 |1          |Alice        |10.0 |low            |
|3       |2024-05-03|200.0 |3          |Charlie      |20.0 |medium         |
|4       |2024-05-04|300.0 |5          |Diana        |30.0 |high           |
|5       |2024-05-10|120.0 |2          |Bob          |12.0 |low            |
+--------+----------+------+-----------+-------------+-----+---------------+
only showing top 5 rows



In [10]:
window = Window.partitionBy("customer_id").orderBy(F.desc("order_date"))

latest_orders = (
    enriched_orders.withColumn("latest_order", F.row_number().over(window))
).filter(F.col("latest_order") == 1)

latest_orders.show(5, truncate=False)

+--------+----------+------+-----------+-------------+-----+---------------+------------+
|order_id|order_date|amount|customer_id|customer_name|bonus|amount_category|latest_order|
+--------+----------+------+-----------+-------------+-----+---------------+------------+
|1       |2024-05-01|100.0 |1          |Alice        |10.0 |low            |1           |
|6       |2024-05-12|250.0 |2          |Bob          |25.0 |high           |1           |
|3       |2024-05-03|200.0 |3          |Charlie      |20.0 |medium         |1           |
|4       |2024-05-04|300.0 |5          |Diana        |30.0 |high           |1           |
+--------+----------+------+-----------+-------------+-----+---------------+------------+



26/03/10 20:29:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 10964570 ms exceeds timeout 120000 ms
26/03/10 20:29:06 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/10 20:44:06 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at